# Blood Pressure Estimation with Neural Networks ~ Resource Profiling Notebook

In this notebook it is possible to find a tutorial on how to personalize a neural-network (NN) on a specific subject.
Specifically, the NN has been pretrained to __quickly__ learn how to predict blood pressure values (SBP/DBP/MAP) from the photoplethysmography (PPG) signal associated with a subject. The netobook shows how to:
> - Load tha physiological signals of a subject from [Vital DB](https://www.nature.com/articles/s41597-022-01411-5) in a PyTorch dataset class
> - Load the pretrained NN (pretrained on the [MIMIC III](https://www.nature.com/articles/sdata201635)), ready for deployment
> - Alternate personalization (namely training) and testing during deployment to predict the Blood Pressure waveform accurately.
Predicting the full BP waveform is a more diffcult task than predicting the BP values as this notebook does. This enables the adoption of smaller models when predicting SBP/DBP/MAP values. This is important for the sake of resource efficiency to prolong the battery duration when the model run on a real device. As the NN undergoes training and testing, it is possible to profile its resource consumptions: given the battery of a real device and the resource consumption of the NN, it is possible to estimate its lifetime. 

![Model Inference Visualization](deployment_overview_no_sig2sig.png)

## Setup Environment

Run the cell below to import the libraries developed in this repository along with other Python/PyTorch packages. The cell belos also sets the device employed to run the notebook (GPU or CPU).

In [ ]:
import os
import sys
project_root = os.path.abspath("..")
folders_to_add = ['data', 'models', 'training_utils']
for folder in folders_to_add:
    sys.path.append(project_root) #if the cell does not run, try to replace this line with 'sys.path.append(os.path.join(project_root, folder))'
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torchinfo import summary
from thop import profile
from models.MAMLLearner import MAMLLearner
from training_utils.helpers import get_encoder_architecture, get_prediction_head_architecture, build_inner_optimizer

# Minimal config for demo
subject_id = 'p001326'
config = {
    "seed": 42,
    "runs_path": f"./data/{subject_id}_runs.pkl",
    "samples_path": f"./data/{subject_id}_samples.npz",
    "model_name": 'ResGruNet',
    "fs": 125,
    "input_seq_len_s": 10,
    "ecg": False, # Set the model to PPG Only!
    "sig2sig": False, # Set the model to predict SBP/DBP/MAP
    "channels": '1, 64, 128, 256',
    "kernel_size": 7,
    "act": 'leaky_relu',
    "pooling": 'avg',
    "embed_dim": 256,
    "num_groups": 8,
    "inner_adapt": 'head',
    "inner_opt": 'adam',
    "inner_head_lr_mult": 1.0,
    "inner_backbone_lr_mult": 0.1,
    "min_run_length": 128,
    "personalization_batch_size": 16,
    "personalization_lr": 1e-2,
    "personalization_steps": 8,
    "criterion": "SmoothL1Loss",
    "grad_clip": 10.0,
    "pretrained_model_ckpt_path": "./models/resgrunet_ppg_no_sig2sig",
}

# Print the loaded configuration
print(config)

# Decide the device to use to run the model
if torch.cuda.is_available():
    # Select the gpu to be usesd
    os.environ['CUDA_VISIBLE_DEVICES'] = "0"
    device = torch._C.device("cuda:0")
    print("Using GPU 0.")
else:
    device = torch.device("cpu")
    print("GPU not available. Using CPU.")

if config['ecg'] or config['sig2sig']:
    raise ValueError("This notebook is intended to be used only with PPG to BP waveform ...")

## Data Loading

Run the two cell below (after running the Setup Environment cells) to load the Vital DB subject into a PyTorch dataset class. Then, a graph display how training and testing are intertwined during deployment. The graph further shows how the BP values (Systolic/Diastolic/Mean Arterial Pressure, the annotation) of the subject vary over time. Note that in general it is difficult to obtain long sequences of valid windows wheree both PPG and BP are not subject to artifacts. We term __run__ a sequence of valid windows with both PPG and BP available. Then, we divide each subject run into blocks that are employed for training (namely, personalization) and testing. 

In [ ]:
def plot_subject_annotation_runs_from_files(
    data,
    runs,
    subject_id="p001326"
):
    """
    Plot annotation statistics for a subject using pre-extracted data and run definitions.

    Parameters
    ----------
    data : np.lib.npyio.NpzFile
        Loaded data for a single subject (as from np.load(samples_path, allow_pickle=True)).
        Must contain 'idxs', 'sbps', 'dbps', 'maps'.
    runs : list of dict
        Each dict contains 'train' and 'test' sample IDs, 'r_idx', and 'b_idx'.
    subject_id : str
        Subject identifier.
    savepath : str
        File path to save the figure.
    show_bp_plot : bool, optional
        If True, show the figure instead of saving. Default=False.
    """

    # === Helper function to extract annotations ===
    def get_annotations(sample_ids):
        sbp_values, dbp_values, map_values = [], [], []
        for sid in sample_ids:
            # Find index of this sample in data['idxs']
            idx_arr = np.where(data['idxs'] == sid)[0]
            idx = int(idx_arr[0])
            sbp_values.append(float(data['sbps'][idx]))
            dbp_values.append(float(data['dbps'][idx]))
            map_values.append(float(data['maps'][idx]))
        return sbp_values, dbp_values, map_values

    # === Collect all data into lists for DataFrame ===
    run_idxs_list, block_list, set_list = [], [], []
    sbp_list, dbp_list, map_list, window_indices = [], [], [], []

    for block in runs:
        # Training
        train_sbp, train_dbp, train_map = get_annotations(block["train"])
        train_indices = [
            int(np.where(data['idxs'] == sid)[0][0])
            for sid in block["train"]
            if sid in data['idxs']
        ]

        run_idxs_list.extend([block["r_idx"]] * len(train_sbp))
        block_list.extend([block["b_idx"]] * len(train_sbp))
        set_list.extend(["training"] * len(train_sbp))
        sbp_list.extend(train_sbp)
        dbp_list.extend(train_dbp)
        map_list.extend(train_map)
        window_indices.extend(train_indices)

        # Testing
        test_sbp, test_dbp, test_map = get_annotations(block["test"])
        test_indices = [
            int(np.where(data['idxs'] == sid)[0][0])
            for sid in block["test"]
            if sid in data['idxs']
        ]

        run_idxs_list.extend([block["r_idx"]] * len(test_sbp))
        block_list.extend([block["b_idx"]] * len(test_sbp))
        set_list.extend(["testing"] * len(test_sbp))
        sbp_list.extend(test_sbp)
        dbp_list.extend(test_dbp)
        map_list.extend(test_map)
        window_indices.extend(test_indices)

    # === Build DataFrame ===
    df = pd.DataFrame({
        "window_index": window_indices,
        "run_idx": run_idxs_list,
        "block": block_list,
        "set": set_list,
        "sbp": sbp_list,
        "dbp": dbp_list,
        "map": map_list,
    }).sort_values(by="window_index").reset_index(drop=True)

    # === Create the plots ===
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    fig.suptitle(f"Subject {subject_id} - Annotation Statistics", fontsize=14, fontweight="bold")

    # Subplot 1 — Window index vs Block number (by Run)
    ax1 = axes[0]
    unique_runs = df["run_idx"].unique()
    colors_runs = plt.cm.tab10(np.linspace(0, 1, len(unique_runs)))
    for i, run_idx in enumerate(unique_runs):
        run_data = df[df["run_idx"] == run_idx]
        ax1.scatter(
            run_data["window_index"], run_data["block"],
            c=[colors_runs[i]], label=f"Run {run_idx}", alpha=0.7, s=30
        )
    ax1.set_ylabel("Block Number")
    ax1.set_title("Window Index vs Block Number")
    ax1.grid(True, alpha=0.3)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    # Subplot 2 — Training vs Testing
    ax2 = axes[1]
    adaptation_data = df[df["set"] == "training"]
    validation_data = df[df["set"] == "testing"]
    ax2.scatter(adaptation_data["window_index"], np.ones(len(adaptation_data)), c="red", label="Training", alpha=0.7, s=30)
    ax2.scatter(validation_data["window_index"], np.ones(len(validation_data))*2, c="blue", label="Testing", alpha=0.7, s=30)
    ax2.set_yticks([1, 2])
    ax2.set_yticklabels(["Training", "Testing"])
    ax2.set_ylabel("Set Type")
    ax2.set_title("Window Index vs Set Type")
    ax2.grid(True, alpha=0.3)
    ax2.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    # Subplot 3 — SBP/DBP/MAP
    ax3 = axes[2]
    ax3.plot(df["window_index"], df["sbp"], "o-", color="red", label="SBP", alpha=0.7, markersize=4, linewidth=1)
    ax3.plot(df["window_index"], df["dbp"], "o-", color="blue", label="DBP", alpha=0.7, markersize=4, linewidth=1)
    ax3.plot(df["window_index"], df["map"], "o-", color="green", label="MAP", alpha=0.7, markersize=4, linewidth=1)
    ax3.set_ylabel("Blood Pressure (mmHg)")
    ax3.set_xlabel("Window Index")
    ax3.set_title("Window Index vs Blood Pressure Values")
    ax3.grid(True, alpha=0.3)
    ax3.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()

    plt.show()


In [ ]:
# Get a subject data: p001326, data contains all samples of a subject, both valid and invalid samples
data = np.load(config['samples_path'], allow_pickle=True)

# Get valid indices for that subject, their indices are recorded in a pickle file
with open(config['runs_path'], "rb") as f:
    runs = pickle.load(f)
print(f"Subject {subject_id} has {len(runs)} blocks of contiguous valid samples, each with {config['personalization_batch_size']} samples for training and {config['personalization_batch_size']} samples for validation")

# Visualization of the runs: a run is a seuquence of contiguous valid windows
plot_subject_annotation_runs_from_files(
    data=data,
    runs=runs,
    subject_id=subject_id
)

In the graph above you can see that between two runs the windows are invalid. Therefore between run 0 and 1 there is no continuity between tha last window of run 0 and the first window of run 1. Time has elapsed between tose two windows and the patient BP values may have shifted (straight line between the two windows shows this event). 

## Model Definition

With the subject run at hand (run both Setup Environment and Data Loading sections is required to continue), in the following two cells we define a model for BP estimation from PPG windows based on NN. Since NNs are notoriously data hungry, we have pretrained the NN on the MIMIC III dataset (this part is not shown here and not relevant for this notebook) to perform deployment with resource-efficient updates. We load it and display a rough estimation of its resource requirements by reporting its number of parameters (in MB) along with the occupation in memory of an input sample for the model (in MB). Importantly, as the model has been pretrained, only part of the model parameters (the prediction head) undergoes actual updates during training, saving computational resources. 

In [ ]:
def get_learner():
    # Initialize architecture: encoder + prediction head
    encoder = get_encoder_architecture(config)
    prediction_head = get_prediction_head_architecture(config)
    learner = MAMLLearner(encoder, prediction_head)

    # Load weights from checkpoint saved during pretraining
    ckpt = torch.load(config['pretrained_model_ckpt_path'], map_location=device)
    learner.load_state_dict(ckpt['learner_state_dict'])
    learner = learner.to(device).eval()
    return learner

In [ ]:
learner = get_learner()

# Define dummy input to print summary
example_shape = (1, config['fs'] * config['input_seq_len_s'])  # Example input
summary(learner, input_size=example_shape)

Note that the above profiling of the model is quite rough and can be further improved. Ideally, the profiling should be fine-grained to report which part of the model requires more storage and computational resources. The profiling should also relate the amount of storage and computation required by the model with the available storage and computational resources of the resource-constrained edge device. For the sake of the demos, it is probably enough to show how many parameters of the whole model are for the prediction head. This is becasue during online training and testing, only the prediction head undergoes updates through gradient descent. Gradient descent is a resource-expensive step of the training phase which should be profiled and compared to the resource required by the testing phase. 

## Deployment

In this final section of the notebook (running all the previous section is required to continue here) we perform the actual deployment of the NN with personalization. For this notebook, where the focus is on resource profiling, the training and testing operations, namely multiply-and-accumulate operations (MACs), should be profiled to estimated the lifetime of the algorithm on a resource-constrained eddge device. As the incoming batches are processed in an online fashion, the personalization loop intertwines training and testing, which require different amount of computing resources. This is the part where more accurate resource profiling on device should be carried out for the estimation fo the algorithm lifetime estimation.  

In [ ]:
def personalization_loop(learner, runs, data, config, device):
    enc, ph = learner.encoder, learner.prediction_head
    enc.eval(); ph.eval()

    mae_per_block, std_per_block = [], []
    train_macs_per_block, val_macs_per_block = [], []

    for block_idx, block in enumerate(runs):
        print(f"\n🟡 Starting block {block_idx+1}/{len(runs)}")

        # --- Adaptation phase ---
        enc.train(); ph.train()
        inner_opt = build_inner_optimizer(enc, ph, base_lr=config['personalization_lr'], config=config)
        B = config['personalization_batch_size']

        # --- Start counter for training MACs ---
        total_train_macs = 0
        train_forward_passes = 0
        
        for step in range(config['personalization_steps']):
            for i in range(0, len(block['train']), B):
                batch_ids = block['train'][i:i+B]

                # --- Build mini-batch ---
                signals_list, targets_list = [], []
                for sid in batch_ids:
                    idx = int(np.where(data['idxs'] == sid)[0])
                    sig = torch.tensor(np.array(data['ppgs'][idx]), dtype=torch.float32).unsqueeze(-1)
                    target = torch.stack([
                        torch.tensor(np.array(data['sbps'][idx])), 
                        torch.tensor(np.array(data['dbps'][idx])), 
                        torch.tensor(np.array(data['maps'][idx]))
                    ], dim=-1)
                    signals_list.append(sig)
                    targets_list.append(target)

                signals = torch.stack(signals_list).to(device)
                targets = torch.stack(targets_list).to(device)
                
                # --- Count MACs for forward pass ---
                # Create a wrapper model for MAC counting
                class ForwardModel(torch.nn.Module):
                    def __init__(self, encoder, pred_head):
                        super().__init__()
                        self.encoder = encoder
                        self.pred_head = pred_head
                    
                    def forward(self, x):
                        return self.pred_head(self.encoder(x))
                
                # Profile only once per batch to avoid overhead
                if train_forward_passes == 0:
                    forward_model = ForwardModel(enc, ph)
                    # Clone input to avoid modifying the original
                    sample_input = signals.clone()
                    macs, params = profile(forward_model, inputs=(sample_input,), verbose=False)
                    # For training, count both forward and backward (backward ≈ 2x forward)
                    macs_per_forward = macs
                    total_train_macs += macs_per_forward * 3  # 1 forward + 2 backward
                else:
                    # Reuse the previously calculated MACs
                    total_train_macs += macs_per_forward * 3
                
                train_forward_passes += 1
                
                outputs = ph(enc(signals))
                loss = F.smooth_l1_loss(outputs, targets) if config['criterion'] == 'SmoothL1Loss' else F.mse_loss(outputs, targets)

                inner_opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(ph.parameters()), config['grad_clip'])
                inner_opt.step()
        
        # --- Stop counter for training MACs ---
        avg_train_macs = total_train_macs / max(train_forward_passes, 1)
        train_macs_per_block.append(total_train_macs)
        print(f"🔧 Training MACs: {total_train_macs / 1e9:.2f} GMACs (avg per batch: {avg_train_macs / 1e9:.2f} GMACs)")
        
        # --- Validation phase ---
        
        # --- Start counter for validation MACs ---
        enc.eval(); ph.eval()
        signals_list, targets_list = [], []
        for sid in block['test']:
            idx = int(np.where(data['idxs'] == sid)[0])
            sig = torch.tensor(np.array(data['ppgs'][idx]), dtype=torch.float32).unsqueeze(-1)
            target = torch.stack([
                torch.tensor(np.array(data['sbps'][idx])), 
                torch.tensor(np.array(data['dbps'][idx])), 
                torch.tensor(np.array(data['maps'][idx]))
            ], dim=-1)
            signals_list.append(sig)
            targets_list.append(target)

        signals = torch.stack(signals_list).to(device)
        targets = torch.stack(targets_list).cpu().numpy()
        
        # --- Count MACs for validation forward pass ---
        with torch.no_grad():
            forward_model = ForwardModel(enc, ph)
            sample_input = signals.clone()
            val_macs, params = profile(forward_model, inputs=(sample_input,), verbose=False)
        
        preds = ph(enc(signals)).detach().cpu().numpy()
        
        # --- Stop counter for validation MACs ---
        val_macs_per_block.append(val_macs)
        print(f"🔍 Validation MACs: {val_macs / 1e6:.2f} MMACs")

        # --- Error calculation ---
        abs_errs = np.abs(preds - targets)
        per_sample_mae = np.mean(abs_errs, axis=1)
        error = float(np.mean(per_sample_mae))
        error_std = float(np.std(per_sample_mae))

        mae_per_block.append(error)
        std_per_block.append(error_std)

        print(f"✅ Block {block_idx+1}: MAE = {error:.3f} ± {error_std:.3f}")

    return {
        'mae_per_block': mae_per_block,
        'std_per_block': std_per_block,
        'train_macs_per_block': train_macs_per_block,
        'val_macs_per_block': val_macs_per_block
    }

In [ ]:
# Always get a fresh learner when running this cell
learner = get_learner()
results = personalization_loop(learner, runs, data, config, device)

In [ ]:
def plot_mae_with_std(mae_per_block, std_per_block, subject_id):
    """
    Plot MAE per block with standard deviation as error bars or shaded area.

    Parameters
    ----------
    mae_per_block : list or np.ndarray
        Mean absolute errors for each block.
    std_per_block : list or np.ndarray
        Standard deviation of absolute errors for each block.
    subject_id : str
        Subject identifier (for title).
    savepath : str, optional
        If provided, save the plot to this path.
    show : bool, default=True
        If True, display the plot instead of saving only.
    """

    mae_per_block = np.array(mae_per_block)
    std_per_block = np.array(std_per_block)
    blocks = np.arange(1, len(mae_per_block) + 1)

    plt.figure(figsize=(12, 7))
    plt.plot(blocks, mae_per_block, marker="o", color="tab:red", label="μ")
    plt.fill_between(
        blocks,
        mae_per_block - std_per_block,
        mae_per_block + std_per_block,
        color="tab:red",
        alpha=0.2,
        label="± σ"
    )

    plt.xlabel("Block Index", fontsize=12)
    plt.ylabel("Mean Absolute Error (mmHg)", fontsize=12)
    plt.title(f"Subject {subject_id} – Online Adaptation MAE (μ ± σ)", fontsize=14, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()

    plt.show()

In [ ]:
# Visualize metric during online learning
plot_mae_with_std(results['mae_per_block'], results['std_per_block'], subject_id)

In [ ]:
def plot_mac_statistics(train_macs_per_block, val_macs_per_block):
    """
    Visualize MAC statistics from personalization loop results.
    
    Args:
        results: Dictionary containing 'train_macs_per_block' and 'val_macs_per_block'
    """
    train_macs = np.array(train_macs_per_block)
    val_macs = np.array(val_macs_per_block)
    
    # Calculate statistics
    total_train_macs = np.sum(train_macs)
    total_val_macs = np.sum(val_macs)
    avg_train_macs = np.mean(train_macs)
    avg_val_macs = np.mean(val_macs)
    
    # Create figure with 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # ==================== Graph 1: Bar Plot for Totals ====================
    categories = ['Total Training\nMACs', 'Total Validation\nMACs']
    values = [total_train_macs / 1e9, total_val_macs / 1e9]  # Convert to GMACs
    colors = ['#e74c3c', '#3498db']
    
    bars = ax1.bar(categories, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels on top of bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.2f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax1.set_ylabel('MACs (GMACs)', fontsize=12, fontweight='bold')
    ax1.set_title('Total MACs: Training vs Validation', fontsize=14, fontweight='bold', pad=20)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    ax1.set_axisbelow(True)
    
    # ==================== Graph 2: Donut Plot for Averages ====================
    avg_total = avg_train_macs + avg_val_macs
    sizes = [avg_train_macs, avg_val_macs]
    labels = ['Training', 'Validation']
    colors_donut = ['#e74c3c', '#3498db']
    explode = (0.05, 0.05)  # Slightly separate both slices
    
    # Create donut plot
    wedges, texts, autotexts = ax2.pie(sizes, labels=labels, colors=colors_donut, 
                                         autopct='%1.1f%%', startangle=90, 
                                         explode=explode, textprops={'fontsize': 12})
    
    # Make percentage text bold and white
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
        autotext.set_fontsize(13)
    
    # Make labels bold
    for text in texts:
        text.set_fontweight('bold')
        text.set_fontsize(12)
    
    # Draw circle in center to make it a donut
    centre_circle = plt.Circle((0, 0), 0.70, fc='white', linewidth=1.5, edgecolor='black')
    ax2.add_artist(centre_circle)
    
    # Add text in center
    ax2.text(0, 0.1, 'Avg MACs\nper Block', ha='center', va='center', 
             fontsize=13, fontweight='bold')
    ax2.text(0, -0.15, f'{avg_total/1e9:.2f} GMACs', ha='center', va='center', 
             fontsize=11, style='italic', color='#555')
    
    ax2.set_title('Average MACs Distribution per Block', fontsize=14, fontweight='bold', pad=20)
    
    # Equal aspect ratio ensures that pie is drawn as a circle
    ax2.axis('equal')
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed statistics
    print("\n" + "="*60)
    print("📊 DETAILED MAC STATISTICS")
    print("="*60)
    print(f"\n🔧 TRAINING:")
    print(f"   Total:   {total_train_macs/1e9:>10.2f} GMACs")
    print(f"   Average: {avg_train_macs/1e9:>10.2f} GMACs per block")
    print(f"   Std Dev: {np.std(train_macs)/1e9:>10.2f} GMACs")
    
    print(f"\n🔍 VALIDATION:")
    print(f"   Total:   {total_val_macs/1e9:>10.2f} GMACs")
    print(f"   Average: {avg_val_macs/1e6:>10.2f} MMACs per block")
    print(f"   Std Dev: {np.std(val_macs)/1e6:>10.2f} MMACs")
    
    print(f"\n📈 COMBINED:")
    print(f"   Total:   {(total_train_macs + total_val_macs)/1e9:>10.2f} GMACs")
    print(f"   Training Percentage: {(total_train_macs/(total_train_macs + total_val_macs)*100):>6.2f}%")
    print(f"   Validation Percentage: {(total_val_macs/(total_train_macs + total_val_macs)*100):>6.2f}%")
    print("="*60 + "\n")

In [ ]:
plot_mac_statistics(results['train_macs_per_block'], results['val_macs_per_block'])